# 2D Ill-Conditioned Ravine Simulation: CLOVES vs. AdamW vs. LION

This notebook demonstrates convergence on an ill-conditioned anisotropic quadratic ravine with condition number $\kappa = 100$:
$$f(x, y) = 0.5 (x^2 + 100 y^2), \quad (x_0, y_0) = (10.0, 1.0)$$

Under extreme valley curvature, AdamW and LION oscillate across the steep $y$-axis walls. CLOVES suppresses transversal oscillations while accelerating down the canyon floor.

In [ ]:
# Install or import CLOVES
try:
    import cloves
except ImportError:
    !pip install -q cloves
    import cloves

import torch
import numpy as np
from cloves import CLOVES_Bisector, CLOVES_Perpendicular
print(f"CLOVES version: {cloves.__version__}")

In [ ]:
def run_simulation(optimizer_cls, opt_kwargs, steps=500):
    point = torch.tensor([10.0, 1.0], requires_grad=True)
    opt = optimizer_cls([point], **opt_kwargs)
    trajectory = [point.detach().clone().numpy()]
    
    for _ in range(steps):
        opt.zero_grad()
        loss = 0.5 * (point[0]**2 + 100.0 * point[1]**2)
        loss.backward()
        opt.step()
        trajectory.append(point.detach().clone().numpy())
        
    return np.array(trajectory), loss.item()

traj_cloves, loss_cloves = run_simulation(CLOVES_Perpendicular, {"lr": 0.02, "q": 1.6, "gamma": 0.10, "gamma_perp": 0.25})
traj_adamw, loss_adamw = run_simulation(torch.optim.AdamW, {"lr": 0.02, "betas": (0.9, 0.999)})

print(f"Final Loss CLOVES: {loss_cloves:.6e}")
print(f"Final Loss AdamW:  {loss_adamw:.6f}")

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 1. Contour map and trajectories
x_grid = np.linspace(-1, 11, 300)
y_grid = np.linspace(-1.5, 1.5, 300)
X, Y = np.meshgrid(x_grid, y_grid)
Z = 0.5 * (X ** 2 + 100.0 * (Y ** 2))

ax1.contour(X, Y, Z, levels=np.logspace(-2, 3, 30), cmap='Blues', alpha=0.5)
ax1.plot(traj_cloves[:, 0], traj_cloves[:, 1], 'purple', lw=2.2, label='CLOVES-Perpendicular')
ax1.plot(traj_adamw[:, 0], traj_adamw[:, 1], 'green', lw=1.5, label='AdamW')
ax1.plot(0, 0, 'k*', markersize=14, label='Minimum (0, 0)')
ax1.set_title('2D Ill-Conditioned Ravine Trajectory')
ax1.set_xlabel('x (slow axis)')
ax1.set_ylabel('y (steep wall axis)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Trajectory y-coordinate oscillation over steps
ax2.plot(traj_adamw[:, 1], 'g-', label='AdamW y-oscillation', alpha=0.7)
ax2.plot(traj_cloves[:, 1], 'm-', label='CLOVES y-damping', lw=2)
ax2.set_title('Transversal Wall Damping Behavior (y-coordinate)')
ax2.set_xlabel('Optimization Step')
ax2.set_ylabel('y-coordinate')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()